# Lezione 05 - RAG Agentico


## Setup

Questo notebook dimostra il modello Agentic RAG (Retrieval-Augmented Generation) utilizzando il Microsoft Agent Framework.

**Prerequisiti:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — il tuo endpoint del servizio Azure AI Search
- `AZURE_SEARCH_API_KEY` — la tua chiave API di Azure AI Search
- Distribuzione di Azure OpenAI configurata tramite variabili di ambiente
- Azure CLI autenticato (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Cos'è Agentic RAG?

Il RAG tradizionale segue una pipeline fissa: recuperare documenti, quindi generare una risposta. **Agentic RAG** va oltre, dando all'agente l'autonomia di decidere **quando** e **come** recuperare le informazioni.

Con Agentic RAG, l'agente può:
- **Decidere** se è necessario il recupero prima di rispondere a una domanda
- **Scegliere** quale fonte di dati o strumento interrogare
- **Valutare** i risultati recuperati ed eseguire recuperi successivi se il primo tentativo non è sufficiente
- **Combinare** informazioni da più passaggi di recupero in una risposta coerente

Questo rende l'agente più flessibile e accurato rispetto a una pipeline statica di recupero e successiva generazione.


## Creazione di uno Strumento di Ricerca

In Agentic RAG, le fonti di dati esterne sono incapsulate come **strumenti** che l'agente può invocare a richiesta. Questo permette all'agente di considerare il recupero dati come un'altra azione che può eseguire, piuttosto che un passaggio obbligatorio.

Di seguito definiamo una base di conoscenza di viaggi e la esponiamo come uno strumento che l'agente può chiamare per cercare informazioni sulle destinazioni.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## Costruire l'agente RAG

Ora creiamo un agente che è istruito a **recuperare sempre informazioni prima di rispondere**. L'agente utilizza lo strumento `search_travel_knowledge` per basare le sue risposte sulla banca dati di conoscenze piuttosto che fare affidamento sui propri dati di addestramento.


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## Recupero Iterativo — Il Modello Maker-Checker

Un vantaggio chiave di Agentic RAG è il **recupero iterativo**. L'agente può eseguire più cicli di ricerca per verificare, perfezionare o ampliare le sue scoperte iniziali — simile a un flusso di lavoro "maker-checker":

1. **Fase Maker**: L'agente recupera informazioni iniziali e redige una risposta.
2. **Fase Checker**: L'agente esegue ulteriori ricerche per verificare dettagli o colmare lacune.

Di seguito, viene posta all'agente una domanda che richiede di confrontare più destinazioni, spingendolo a cercare più volte.


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## Sommario

In questa lezione hai imparato come costruire un sistema **Agentic RAG** utilizzando il Microsoft Agent Framework:

- **Agentic RAG** permette agli agenti di decidere autonomamente quando recuperare informazioni, rendendo il recupero dinamico anziché fisso.
- **Strumenti come sorgenti di dati**: basi di conoscenza esterne (come Azure AI Search) sono incapsulate come strumenti che l'agente può invocare.
- **Recupero iterativo**: il modello maker-checker consente all'agente di eseguire più cicli di recupero — cercando, verificando e raffinando — prima di produrre una risposta finale.

In produzione, sostituiresti il `TRAVEL_KNOWLEDGE_BASE` in memoria con un vero indice Azure AI Search per gestire il recupero su larga scala di documenti di viaggio.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:  
Questo documento è stato tradotto utilizzando il servizio di traduzione automatica [Co-op Translator](https://github.com/Azure/co-op-translator). Pur impegnandoci per garantire precisione, si prega di considerare che le traduzioni automatiche possono contenere errori o inesattezze. Il documento originale nella sua lingua nativa deve essere considerato la fonte autorevole. Per informazioni critiche, si raccomanda la traduzione professionale effettuata da un essere umano. Non siamo responsabili per eventuali fraintendimenti o interpretazioni errate derivanti dall’uso di questa traduzione.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
